# Setup & Load Master Data

In [3]:
import re
import pandas as pd
from sqlalchemy import create_engine
from rapidfuzz import fuzz, process

engine = create_engine("postgresql://tiarasabrina@localhost:5432/wilayah_db")
wilayah = pd.read_sql("SELECT kode, nama FROM wilayah.wilayah", engine)
wilayah["nama"] = wilayah["nama"].str.upper().str.strip()
wilayah = wilayah.dropna(subset=["kode", "nama"])

kodepos_df = pd.read_sql("SELECT kode, kodepos FROM wilayah_kodepos.wilayah_kodepos", engine)
kodepos_df["kodepos"] = kodepos_df["kodepos"].astype(str).str.strip()

print(wilayah.shape, kodepos_df.shape)
wilayah.head()

(91599, 2) (83762, 2)


,kode,nama
0,11,ACEH
1,11.01,KABUPATEN ACEH SELATAN
2,11.01.01,BAKONGAN
3,11.01.01.2001,KEUDE BAKONGAN
4,11.01.01.2002,UJONG MANGKI


# Get Level & Split Per Level

In [4]:
def get_level(kode):
    parts = kode.split(".")
    return {1: "provinsi", 2: "kabkota", 3: "kecamatan", 4: "desa"}.get(len(parts), "unknown")

wilayah["level"] = wilayah["kode"].apply(get_level)
df_prov    = wilayah[wilayah["level"] == "provinsi"].reset_index(drop=True)
df_kabkota = wilayah[wilayah["level"] == "kabkota"].reset_index(drop=True)
df_kec     = wilayah[wilayah["level"] == "kecamatan"].reset_index(drop=True)
df_desa    = wilayah[wilayah["level"] == "desa"].reset_index(drop=True)

print(f"provinsi={len(df_prov)}, kabkota={len(df_kabkota)}, kecamatan={len(df_kec)}, desa={len(df_desa)}")

provinsi=38, kabkota=514, kecamatan=7285, desa=83762


# Core Name

In [5]:
ADMIN_WORDS = {"KOTA", "KABUPATEN", "KAB", "ADMINISTRASI", "ADM"}

def core_name(nama):
    tokens = nama.replace(".", " ").split()
    return " ".join(t for t in tokens if t not in ADMIN_WORDS).strip()

for df_ in (df_prov, df_kabkota, df_kec, df_desa):
    df_["nama_core"] = df_["nama"].apply(core_name)

# test cepat
print(core_name("KOTA ADMINISTRASI JAKARTA SELATAN"))
df_kabkota[df_kabkota["nama"].str.contains("JAKARTA SELATAN")][["nama", "nama_core"]]

JAKARTA SELATAN


,nama,nama_core
158,KOTA ADMINISTRASI JAKARTA SELATAN,JAKARTA SELATAN


# Dictionary

In [6]:
PROVINCE_ALIASES = {
    "DKI JAKARTA": "DKI JAKARTA", "DKI": "DKI JAKARTA", "JKT": "DKI JAKARTA",
    "JAWA BARAT": "JAWA BARAT", "JABAR": "JAWA BARAT",
    "JAWA TENGAH": "JAWA TENGAH", "JATENG": "JAWA TENGAH",
    "JAWA TIMUR": "JAWA TIMUR", "JATIM": "JAWA TIMUR",
    "KALIMANTAN SELATAN": "KALIMANTAN SELATAN", "KALSEL": "KALIMANTAN SELATAN",
    "KALIMANTAN TIMUR": "KALIMANTAN TIMUR", "KALTIM": "KALIMANTAN TIMUR",
    "KALIMANTAN BARAT": "KALIMANTAN BARAT", "KALBAR": "KALIMANTAN BARAT",
    "KALIMANTAN TENGAH": "KALIMANTAN TENGAH", "KALTENG": "KALIMANTAN TENGAH",
    "KALIMANTAN UTARA": "KALIMANTAN UTARA", "KALTARA": "KALIMANTAN UTARA",
    "SUMATERA UTARA": "SUMATERA UTARA", "SUMUT": "SUMATERA UTARA",
    "SUMATERA BARAT": "SUMATERA BARAT", "SUMBAR": "SUMATERA BARAT",
    "SUMATERA SELATAN": "SUMATERA SELATAN", "SUMSEL": "SUMATERA SELATAN",
    "SULAWESI SELATAN": "SULAWESI SELATAN", "SULSEL": "SULAWESI SELATAN",
    "SULAWESI UTARA": "SULAWESI UTARA", "SULUT": "SULAWESI UTARA",
    "SULAWESI TENGAH": "SULAWESI TENGAH", "SULTENG": "SULAWESI TENGAH",
    "SULAWESI TENGGARA": "SULAWESI TENGGARA", "SULTRA": "SULAWESI TENGGARA",
    "SULAWESI BARAT": "SULAWESI BARAT", "SULBAR": "SULAWESI BARAT",
    "NUSA TENGGARA BARAT": "NUSA TENGGARA BARAT", "NTB": "NUSA TENGGARA BARAT",
    "NUSA TENGGARA TIMUR": "NUSA TENGGARA TIMUR", "NTT": "NUSA TENGGARA TIMUR",
    "KEPULAUAN RIAU": "KEPULAUAN RIAU", "KEPRI": "KEPULAUAN RIAU",
    "KEPULAUAN BANGKA BELITUNG": "KEPULAUAN BANGKA BELITUNG", "BABEL": "KEPULAUAN BANGKA BELITUNG",
    "DAERAH ISTIMEWA YOGYAKARTA": "DAERAH ISTIMEWA YOGYAKARTA",
    "DIY": "DAERAH ISTIMEWA YOGYAKARTA", "YOGYA": "DAERAH ISTIMEWA YOGYAKARTA", "JOGJA": "DAERAH ISTIMEWA YOGYAKARTA",
    "PAPUA BARAT": "PAPUA BARAT", "PAPBAR": "PAPUA BARAT",
}
CITY_ALIASES = {
    "JAKARTA SELATAN": "JAKARTA SELATAN", "JAKSEL": "JAKARTA SELATAN",
    "JAKARTA PUSAT": "JAKARTA PUSAT", "JAKPUS": "JAKARTA PUSAT",
    "JAKARTA UTARA": "JAKARTA UTARA", "JAKUT": "JAKARTA UTARA",
    "JAKARTA BARAT": "JAKARTA BARAT", "JAKBAR": "JAKARTA BARAT",
    "JAKARTA TIMUR": "JAKARTA TIMUR", "JAKTIM": "JAKARTA TIMUR",
    "BANDUNG": "BANDUNG", "BDG": "BANDUNG", "SURABAYA": "SURABAYA", "SBY": "SURABAYA",
    "SEMARANG": "SEMARANG", "SMG": "SEMARANG", "MEDAN": "MEDAN", "MDN": "MEDAN",
    "MAKASSAR": "MAKASSAR", "MKS": "MAKASSAR", "PALEMBANG": "PALEMBANG", "PLB": "PALEMBANG",
    "TANGERANG": "TANGERANG", "TGRG": "TANGERANG", "TNGRG": "TANGERANG",
    "BEKASI": "BEKASI", "BKS": "BEKASI", "DEPOK": "DEPOK", "DPK": "DEPOK",
    "BOGOR": "BOGOR", "BGR": "BOGOR",
}
_ALL_ALIASES = {**CITY_ALIASES, **PROVINCE_ALIASES}
_ALIAS_KEYS_SORTED = sorted(_ALL_ALIASES.keys(), key=len, reverse=True)
_ALIAS_PATTERN = re.compile(r"\b(" + "|".join(re.escape(k) for k in _ALIAS_KEYS_SORTED) + r")\b")

def expand_abbreviations(text):
    return _ALIAS_PATTERN.sub(lambda m: _ALL_ALIASES[m.group(1)], text)

# test
print(expand_abbreviations("JAKSEL DAN JATENG"))

JAKARTA SELATAN DAN JAWA TENGAH


# Pattern Structural

In [105]:
# Pattern Build

STREET_PATTERN   = r"\b(JALAN|JL|JLN)\b\.?\s*"
BUILDING_PATTERN = r"\b(GEDUNG|GD|GDG|GED|MENARA|MNR|TOWER|TWR|WISMA|GRAHA|GRIYA|PLAZA|MALL|RUKO|RUKAN|BUILDING|OFFICE|SUITE|KOMPLEK|KOMPLEKS|PERKANTORAN|APARTEMEN|APARTMENT|CENTRE|CENTER|SENTRA|PARK|SQUARE)\b\.?"
FLOOR_PATTERN    = r"\b(LANTAI|LT|LTAI|FLOOR|FLR)\b\.?\s*([A-Z0-9\-]+)"
NO_PATTERN       = r"\b(NOMOR|NMR|NO)\b\.?\s*([A-Z0-9\-\/]+)"
BLOK_PATTERN     = r"\bBLOK\b\.?\s*([A-Z0-9\-]+)"
KAV_PATTERN      = r"\bKAV(?:LING)?\b\.?\s*([A-Z0-9\-]+)"
POSTAL_PATTERN   = r"(?<!\d)(\d{5})(?!\d)"
RT_RW_PATTERNS = [
    r"\bRT\s*/?\s*RW\b\.?\s*(\d+)\s*/\s*(\d+)",
    r"\bRT\b\.?\s*(\d+)\s*/\s*RW\b\.?\s*(\d+)",
    r"\bRT\b\.?\s*(\d+)\D{0,15}RW\b\.?\s*(\d+)",
    r"\bRT\b\.?\s*(\d+)\s*/\s*(\d+)\b",
]

In [106]:
def tokenize(text):
    return re.findall(r"[A-Z0-9]+", text)

In [107]:
remaining = "JL BLOK M NO. 12 LT 17 BLOK KV, RT 02RW 07 GD SENTRAYA MARYA JAKARTA 12950- INDONESIA , JAKARTA SELATAN".upper()

def extract_and_remove_kodepos(text):
    m = re.search(POSTAL_PATTERN, text)
    if not m:
        return None, text
    value = m.group(1)
    new_text = text[:m.start()] + text[m.end():]
    return value, new_text

kodepos_hasil, remaining = extract_and_remove_kodepos(remaining)
print("kodepos:", kodepos_hasil)
print("remaining:", remaining)

kodepos: 12950
remaining: JL BLOK M NO. 12 LT 17 BLOK KV, RT 02RW 07 GD SENTRAYA MARYA JAKARTA - INDONESIA , JAKARTA SELATAN


In [108]:
def extract_and_remove_rt_rw(text):
    for p in RT_RW_PATTERNS:
        m = re.search(p, text)
        if m:
            rt, rw = m.group(1), m.group(2)
            new_text = text[:m.start()] + text[m.end():]
            return rt, rw, new_text
    return None, None, text

rt_hasil, rw_hasil, remaining = extract_and_remove_rt_rw(remaining)
print("rt:", rt_hasil, "| rw:", rw_hasil)
print("remaining:", remaining)

rt: 02 | rw: 07
remaining: JL BLOK M NO. 12 LT 17 BLOK KV,  GD SENTRAYA MARYA JAKARTA - INDONESIA , JAKARTA SELATAN


In [109]:
def extract_and_remove_no(text):
    m = re.search(NO_PATTERN, text)
    if not m:
        return None, text
    value = m.group(2)
    new_text = text[:m.start()] + text[m.end():]
    return value, new_text

no_hasil, remaining = extract_and_remove_no(remaining)
print("no:", no_hasil)
print("remaining:", remaining)

no: 12
remaining: JL BLOK M  LT 17 BLOK KV,  GD SENTRAYA MARYA JAKARTA - INDONESIA , JAKARTA SELATAN


In [110]:
def extract_and_remove_lantai(text):
    m = re.search(FLOOR_PATTERN, text)
    if not m:
        return None, text
    value = m.group(2)
    new_text = text[:m.start()] + text[m.end():]
    return value, new_text

lantai_hasil, remaining = extract_and_remove_lantai(remaining)
print("lantai:", lantai_hasil)
print("remaining:", remaining)

lantai: 17
remaining: JL BLOK M   BLOK KV,  GD SENTRAYA MARYA JAKARTA - INDONESIA , JAKARTA SELATAN


In [111]:
known_wilayah_tokens = set(tokenize(" ".join(df_prov["nama_core"]))) | set(tokenize(" ".join(df_kabkota["nama_core"])))
struktural_words = {"NO", "NOMOR", "NMR", "LT", "LANTAI", "LTAI", "FLOOR", "FLR", "RT", "RW", "BLOK", "KAV", "KAVLING"}

def extract_and_remove_gedung(text):
    pattern = re.compile(
        r"\b(?:GEDUNG|GD|GDG|GED|MENARA|MNR|TOWER|TWR|WISMA|GRAHA|GRIYA|PLAZA|MALL|RUKO|RUKAN|BUILDING|OFFICE|SUITE|KOMPLEK|KOMPLEKS|PERKANTORAN|APARTEMEN|APARTMENT|CENTRE|CENTER|SENTRA|PARK|SQUARE)\b\.?\s*"
        r"([A-Z0-9][A-Z0-9\-]*(?:\s+[A-Z0-9][A-Z0-9\-]*){0,2})"
    )
    m = pattern.search(text)
    if not m:
        return None, text

    words = m.group(1).split()
    cut_index = len(words)
    for i, w in enumerate(words):
        if w in known_wilayah_tokens and len(w) >= 4:
            cut_index = min(cut_index, i)
            break
    for i, w in enumerate(words[:cut_index]):
        if w in struktural_words:
            cut_index = min(cut_index, i)
            break

    nama = " ".join(words[:cut_index]).strip(" .,-")
    if not nama:
        return None, text

    # hapus SELURUH match keyword+nama dari teks (bukan cuma nama-nya)
    full_match_text = m.group(0)
    consumed = full_match_text[:len(m.group(0)) - len(m.group(1))] + nama
    new_text = text.replace(consumed, "", 1)
    return nama, new_text

gedung_hasil, remaining = extract_and_remove_gedung(remaining)
print("gedung:", gedung_hasil)
print("remaining:", remaining)

gedung: SENTRAYA MARYA
remaining: JL BLOK M   BLOK KV,   JAKARTA - INDONESIA , JAKARTA SELATAN


In [113]:
def extract_and_remove_jalan(text):
    pattern = re.compile(
        r"\b(?:JALAN|JL|JLN)\b\.?\s*"
        r"([A-Z0-9][A-Z0-9\-]*(?:\s+[A-Z0-9][A-Z0-9\-]*){0,3})"
    )
    m = pattern.search(text)
    if not m:
        return None, text

    words = m.group(1).split()

    cut_index = len(words)

    # cek known wilayah tokens -- boleh cut di index manapun, termasuk index 0
    for i, w in enumerate(words):
        if w in known_wilayah_tokens and len(w) >= 4:
            cut_index = min(cut_index, i)
            break

    # cek struktural words -- SKIP index 0, karena kata pertama boleh jadi bagian nama asli
    # (misal "Blok M" -- "BLOK" di index 0 itu sah, bukan penanda field terpisah)
    for i, w in enumerate(words[:cut_index]):
        if i == 0:
            continue
        if w in struktural_words:
            cut_index = min(cut_index, i)
            break

    nama = " ".join(words[:cut_index]).strip(" .,-")
    if not nama:
        return None, text

    full_match_text = m.group(0)
    consumed = full_match_text[:len(m.group(0)) - len(m.group(1))] + nama
    new_text = text.replace(consumed, "", 1)
    return nama, new_text

jalan_hasil, remaining = extract_and_remove_jalan(remaining)
print("jalan:", jalan_hasil)
print("remaining:", remaining)

jalan: BLOK M
remaining:    BLOK KV,   JAKARTA - INDONESIA , JAKARTA SELATAN


In [114]:
def extract_and_remove_blok(text):
    m = re.search(BLOK_PATTERN, text)
    if not m:
        return None, text
    value = m.group(1)
    new_text = text[:m.start()] + text[m.end():]
    return value, new_text

blok_hasil, remaining = extract_and_remove_blok(remaining)
print("blok:", blok_hasil)
print("remaining:", remaining)

blok: KV
remaining:    ,   JAKARTA - INDONESIA , JAKARTA SELATAN


In [115]:
def extract_and_remove_kav(text):
    m = re.search(KAV_PATTERN, text)
    if not m:
        return None, text
    value = m.group(1)
    new_text = text[:m.start()] + text[m.end():]
    return value, new_text

kav_hasil, remaining = extract_and_remove_kav(remaining)
print("kav:", kav_hasil)
print("remaining:", remaining)

kav: None
remaining:    ,   JAKARTA - INDONESIA , JAKARTA SELATAN


In [116]:
# Clean leftover function

def clean_leftover(s):
    s = re.sub(NO_PATTERN, "", s)
    s = re.sub(FLOOR_PATTERN, "", s)
    s = re.sub(BLOK_PATTERN, "", s)
    s = re.sub(KAV_PATTERN, "", s)
    for p in RT_RW_PATTERNS:
        s = re.sub(p, "", s)
    return re.sub(r"\s+", " ", s).strip(" .,-")

print(clean_leftover(remaining))

JAKARTA - INDONESIA , JAKARTA SELATAN


In [128]:
def sequence_in_tokens(cand_tokens, text_tokens):
    n, m = len(cand_tokens), len(text_tokens)
    if n == 0 or n > m:
        return False
    for i in range(m - n + 1):
        if text_tokens[i:i+n] == cand_tokens:
            return True
    return False

In [126]:
def exact_match_safe(text_tokens, candidates_df, min_len=4, name_col="nama_core"):
    text_tokens_set = set(text_tokens)
    cand_sorted = candidates_df.assign(_len=candidates_df[name_col].str.len()).sort_values("_len", ascending=False)
    for _, row in cand_sorted.iterrows():
        core = row[name_col]
        if not core or len(core) < min_len:
            continue
        cand_tokens = tokenize(core)
        if not cand_tokens:
            continue
        if sequence_in_tokens(cand_tokens, text_tokens):
            return row["nama"], row["kode"]
        cand_nospace = "".join(cand_tokens)
        if len(cand_tokens) > 1 and cand_nospace in text_tokens_set and len(cand_nospace) >= min_len:
            return row["nama"], row["kode"]
    return None, None

nama_kab, kode_kab = exact_match_safe(text_tokens, df_kabkota, min_len=4)
print(nama_kab, kode_kab)

KOTA ADMINISTRASI JAKARTA SELATAN 31.74


In [129]:
FUZZY_THRESHOLD = 95

def fuzzy_match_one(text, candidates_df, score_cutoff=FUZZY_THRESHOLD):
    if candidates_df.empty:
        return None, None, 0
    match = process.extractOne(text, candidates_df["nama"], scorer=fuzz.token_set_ratio, score_cutoff=score_cutoff)
    if not match:
        return None, None, 0
    nama, score, idx = match
    row = candidates_df.loc[idx]
    if isinstance(row, pd.DataFrame):
        row = row.iloc[0]
    return row["nama"], row["kode"], score

In [124]:
STRICT_THRESHOLD = 98

def match_wilayah_final(leftover_text, kodepos=None):
    tokens = tokenize(expand_abbreviations(leftover_text))
    result = {
        "provinsi": None, "kode_prov": None, "score_prov": 0,
        "kabkota": None, "kode_kabkota": None, "score_kabkota": 0,
        "kecamatan": None, "kode_kec": None, "score_kec": 0,
        "desa": None, "kode_desa": None, "score_desa": 0,
        "match_path": None,
    }

    # ============================================
    # PATH A: ADA KODEPOS
    # ============================================
    if kodepos:
        kandidat_kode = kodepos_df.loc[kodepos_df["kodepos"] == str(kodepos), "kode"].tolist()
        if kandidat_kode:
            print(f"kodepos '{kodepos}' -> {len(kandidat_kode)} kandidat kode wilayah")
            result["match_path"] = "with_postcode"

            pool_desa = df_desa[df_desa["kode"].isin(kandidat_kode)]
            nama_d, kode_d = exact_match_safe(tokens, pool_desa, min_len=4)
            score_d = 100
            if not nama_d:
                nama_d, kode_d, score_d = fuzzy_match_one(leftover_text, pool_desa, score_cutoff=STRICT_THRESHOLD)

            if nama_d:
                print(f"desa ketemu: {nama_d} (skor={score_d})")
                parts = kode_d.split(".")
                result["desa"], result["kode_desa"], result["score_desa"] = nama_d, kode_d, score_d
                row_kec = df_kec[df_kec["kode"] == ".".join(parts[:3])]
                row_kab = df_kabkota[df_kabkota["kode"] == ".".join(parts[:2])]
                row_prov = df_prov[df_prov["kode"] == parts[0]]
                if not row_kec.empty:
                    result["kecamatan"], result["kode_kec"], result["score_kec"] = row_kec.iloc[0]["nama"], ".".join(parts[:3]), score_d
                if not row_kab.empty:
                    result["kabkota"], result["kode_kabkota"], result["score_kabkota"] = row_kab.iloc[0]["nama"], ".".join(parts[:2]), score_d
                if not row_prov.empty:
                    result["provinsi"], result["kode_prov"], result["score_prov"] = row_prov.iloc[0]["nama"], parts[0], score_d
            else:
                print("desa gak ketemu dalam pool kodepos -> kosongin aja")
            return result
        else:
            print(f"kodepos '{kodepos}' tidak ditemukan di tabel -> lanjut sebagai tanpa kodepos")

    # ============================================
    # PATH B: TANPA KODEPOS (atau kodepos gak valid)
    # Tiap level: kalau parent ketemu -> cari dipersempit. Kalau parent kosong -> cari GLOBAL.
    # Gak pernah stop total, selalu lanjut ke level berikutnya.
    # ============================================
    result["match_path"] = "without_postcode"

    # LEVEL 1: PROVINSI (selalu global)
    nama_p, kode_p = exact_match_safe(tokens, df_prov, min_len=4)
    score_p = 100
    if not nama_p:
        nama_p, kode_p, score_p = fuzzy_match_one(leftover_text, df_prov, score_cutoff=STRICT_THRESHOLD)
    if nama_p:
        result["provinsi"], result["kode_prov"], result["score_prov"] = nama_p, kode_p, score_p
        print(f"provinsi ketemu: {nama_p} (skor={score_p})")
    else:
        print("provinsi: tidak ketemu -> lanjut cari kab/kota secara GLOBAL")

    # LEVEL 2: KAB/KOTA -> persempit kalau provinsi ketemu, else global
    kandidat_kab = df_kabkota[df_kabkota["kode"].str.startswith(kode_p + ".")] if nama_p else df_kabkota
    nama_k, kode_k = exact_match_safe(tokens, kandidat_kab, min_len=4)
    score_k = 100
    if not nama_k:
        nama_k, kode_k, score_k = fuzzy_match_one(leftover_text, kandidat_kab, score_cutoff=STRICT_THRESHOLD)
    if nama_k:
        result["kabkota"], result["kode_kabkota"], result["score_kabkota"] = nama_k, kode_k, score_k
        print(f"kab/kota ketemu: {nama_k} (skor={score_k})")
        # kalau provinsi tadinya kosong, derive dari kab/kota yang ketemu
        if not nama_p:
            kp_derived = kode_k.split(".")[0]
            row_prov = df_prov[df_prov["kode"] == kp_derived]
            if not row_prov.empty:
                result["provinsi"], result["kode_prov"], result["score_prov"] = row_prov.iloc[0]["nama"], kp_derived, score_k
                print(f"  -> provinsi di-derive dari kab/kota: {result['provinsi']}")
    else:
        print("kab/kota: tidak ketemu -> lanjut cari kecamatan secara GLOBAL")

    # LEVEL 3: KECAMATAN -> persempit kalau kab/kota ketemu, else global
    kandidat_kec = df_kec[df_kec["kode"].str.startswith(kode_k + ".")] if nama_k else df_kec
    nama_c, kode_c = exact_match_safe(tokens, kandidat_kec, min_len=4)
    score_c = 100
    if not nama_c:
        nama_c, kode_c, score_c = fuzzy_match_one(leftover_text, kandidat_kec, score_cutoff=STRICT_THRESHOLD)
    if nama_c:
        result["kecamatan"], result["kode_kec"], result["score_kec"] = nama_c, kode_c, score_c
        print(f"kecamatan ketemu: {nama_c} (skor={score_c})")
        # derive kab/kota & provinsi kalau masih kosong
        if not nama_k:
            kk_derived = ".".join(kode_c.split(".")[:2])
            row_kab = df_kabkota[df_kabkota["kode"] == kk_derived]
            if not row_kab.empty:
                result["kabkota"], result["kode_kabkota"], result["score_kabkota"] = row_kab.iloc[0]["nama"], kk_derived, score_c
                print(f"  -> kab/kota di-derive dari kecamatan: {result['kabkota']}")
            if not result["provinsi"]:
                kp_derived = kode_c.split(".")[0]
                row_prov = df_prov[df_prov["kode"] == kp_derived]
                if not row_prov.empty:
                    result["provinsi"], result["kode_prov"], result["score_prov"] = row_prov.iloc[0]["nama"], kp_derived, score_c
                    print(f"  -> provinsi di-derive dari kecamatan: {result['provinsi']}")
    else:
        print("kecamatan: tidak ketemu -> lanjut cari desa secara GLOBAL")

    # LEVEL 4: DESA -> persempit kalau kecamatan ketemu, else global
    kandidat_desa = df_desa[df_desa["kode"].str.startswith(kode_c + ".")] if nama_c else df_desa
    nama_d, kode_d = exact_match_safe(tokens, kandidat_desa, min_len=4)
    score_d = 100
    if not nama_d:
        nama_d, kode_d, score_d = fuzzy_match_one(leftover_text, kandidat_desa, score_cutoff=STRICT_THRESHOLD)
    if nama_d:
        result["desa"], result["kode_desa"], result["score_desa"] = nama_d, kode_d, score_d
        print(f"desa ketemu: {nama_d} (skor={score_d})")
        # derive semua di atasnya kalau masih kosong
        parts = kode_d.split(".")
        if not result["kecamatan"]:
            row = df_kec[df_kec["kode"] == ".".join(parts[:3])]
            if not row.empty:
                result["kecamatan"], result["kode_kec"], result["score_kec"] = row.iloc[0]["nama"], ".".join(parts[:3]), score_d
        if not result["kabkota"]:
            row = df_kabkota[df_kabkota["kode"] == ".".join(parts[:2])]
            if not row.empty:
                result["kabkota"], result["kode_kabkota"], result["score_kabkota"] = row.iloc[0]["nama"], ".".join(parts[:2]), score_d
        if not result["provinsi"]:
            row = df_prov[df_prov["kode"] == parts[0]]
            if not row.empty:
                result["provinsi"], result["kode_prov"], result["score_prov"] = row.iloc[0]["nama"], parts[0], score_d
    else:
        print("desa: tidak ketemu -> kosongin aja")

    return result


# test
leftover_test = "64472 NGANJUK JAWA TIMUR"
hasil = match_wilayah_final(leftover_test, kodepos=None)
print("\nHASIL:", hasil)

provinsi ketemu: JAWA TIMUR (skor=100)
kab/kota ketemu: KABUPATEN NGANJUK (skor=100)
kecamatan ketemu: NGANJUK (skor=100)
desa: tidak ketemu -> kosongin aja

HASIL: {'provinsi': 'JAWA TIMUR', 'kode_prov': '35', 'score_prov': 100, 'kabkota': 'KABUPATEN NGANJUK', 'kode_kabkota': '35.18', 'score_kabkota': 100, 'kecamatan': 'NGANJUK', 'kode_kec': '35.18.13', 'score_kec': 100, 'desa': None, 'kode_desa': None, 'score_desa': 0, 'match_path': 'without_postcode'}


In [138]:
import re

STRICT_THRESHOLD = 98

# kata generik yang gak usah dipake buat nyari & buang token
GENERIC_PREFIXES = {"KABUPATEN", "KAB", "KOTA", "KECAMATAN", "KEC", "DESA", "KELURAHAN", "KEL"}


def remove_matched_text(text, matched_name):
    """
    Buang kata2 dari `matched_name` yang beneran muncul di `text`, per-kata (bukan frasa utuh).
    Fuzzy match sering matched candidate name-nya "KABUPATEN NGANJUK" padahal teks aslinya
    cuma ada kata "NGANJUK" doang -> kalau strip literal frasa utuh, gak ketemu & gak kehapus
    -> kata itu bisa "ke-reuse" di level berikutnya (double match).
    """
    if not matched_name:
        return text
    words = [w for w in re.split(r'\s+', matched_name.strip()) if w and w.upper() not in GENERIC_PREFIXES]
    cleaned = text
    for w in words:
        pattern = r'\b' + re.escape(w) + r'\b'
        cleaned = re.sub(pattern, ' ', cleaned, flags=re.IGNORECASE)
    return re.sub(r'\s+', ' ', cleaned).strip()


def common_kode_prefix(kode_list):
    """
    Cari level kode (prov/kab/kec/desa) yang SAMA di semua kandidat kode.
    len==1 -> cuma prov sama, len==2 -> prov+kab sama, len==3 -> +kec sama, dst.
    """
    splits = [k.split(".") for k in kode_list]
    max_level = min(len(s) for s in splits)
    common = []
    for i in range(max_level):
        vals = set(s[i] for s in splits)
        if len(vals) == 1:
            common.append(splits[0][i])
        else:
            break
    return common


def find_desa_bigram_match(tokens, desa_pool, min_len=6):
    """
    Cek apakah ada 2 token BERURUTAN di `tokens` yang kalau digabung tanpa spasi
    match EXACT ke salah satu nama desa di `desa_pool`.
    Ini nangkep kasus "PACE KULON" (2 kata di teks user) vs "PACEKULON" (1 kata di data),
    yang tanpa ini bakal ke-grab duluan sama level kecamatan (krn "PACE" match kecamatan).
    Return (nama_desa, kode_desa) atau (None, None).
    """
    if desa_pool.empty or len(tokens) < 2:
        return None, None
    names_map = {str(n).upper(): (n, k) for n, k in zip(desa_pool["nama"], desa_pool["kode"])}
    for i in range(len(tokens) - 1):
        joined = (tokens[i] + tokens[i + 1]).upper()
        if len(joined) >= min_len and joined in names_map:
            return names_map[joined]
    return None, None


def match_wilayah_final(leftover_text, kodepos=None):
    tokens = tokenize(expand_abbreviations(leftover_text))
    result = {
        "provinsi": None, "kode_prov": None, "score_prov": 0,
        "kabkota": None, "kode_kabkota": None, "score_kabkota": 0,
        "kecamatan": None, "kode_kec": None, "score_kec": 0,
        "desa": None, "kode_desa": None, "score_desa": 0,
        "match_path": None,
    }

    # ============================================
    # PATH A: ADA KODEPOS
    # ============================================
    if kodepos:
        kandidat_kode = kodepos_df.loc[kodepos_df["kodepos"] == str(kodepos), "kode"].tolist()
        if kandidat_kode:
            print(f"kodepos '{kodepos}' -> {len(kandidat_kode)} kandidat kode wilayah")
            result["match_path"] = "with_postcode"

            pool_desa = df_desa[df_desa["kode"].isin(kandidat_kode)]
            nama_d, kode_d = exact_match_safe(tokens, pool_desa, min_len=4)
            score_d = 100
            if not nama_d:
                nama_d, kode_d = find_desa_bigram_match(tokens, pool_desa)
                if nama_d:
                    score_d = 100
                    print(f"desa ketemu via bigram: {nama_d} (skor={score_d})")
            if not nama_d:
                nama_d, kode_d, score_d = fuzzy_match_one(leftover_text, pool_desa, score_cutoff=STRICT_THRESHOLD)

            if nama_d:
                print(f"desa ketemu: {nama_d} (skor={score_d})")
                parts = kode_d.split(".")
                result["desa"], result["kode_desa"], result["score_desa"] = nama_d, kode_d, score_d
                row_kec = df_kec[df_kec["kode"] == ".".join(parts[:3])]
                row_kab = df_kabkota[df_kabkota["kode"] == ".".join(parts[:2])]
                row_prov = df_prov[df_prov["kode"] == parts[0]]
                if not row_kec.empty:
                    result["kecamatan"], result["kode_kec"], result["score_kec"] = row_kec.iloc[0]["nama"], ".".join(parts[:3]), score_d
                if not row_kab.empty:
                    result["kabkota"], result["kode_kabkota"], result["score_kabkota"] = row_kab.iloc[0]["nama"], ".".join(parts[:2]), score_d
                if not row_prov.empty:
                    result["provinsi"], result["kode_prov"], result["score_prov"] = row_prov.iloc[0]["nama"], parts[0], score_d
            else:
                # FALLBACK: desa gak ketemu, tapi kandidat_kode dari kodepos
                # kemungkinan besar semua satu kecamatan/kabkota/provinsi yang sama.
                print("desa gak ketemu -> coba derive kec/kab/prov dari kesamaan kandidat kodepos")
                common = common_kode_prefix(kandidat_kode)
                DERIVED_SCORE = 90  # penanda: derived dari postcode only, bukan exact/fuzzy match teks

                if len(common) >= 3:
                    kode_kec_common = ".".join(common[:3])
                    row_kec = df_kec[df_kec["kode"] == kode_kec_common]
                    if not row_kec.empty:
                        result["kecamatan"], result["kode_kec"], result["score_kec"] = row_kec.iloc[0]["nama"], kode_kec_common, DERIVED_SCORE
                        print(f"  -> kecamatan di-derive dari kodepos: {result['kecamatan']}")
                if len(common) >= 2:
                    kode_kab_common = ".".join(common[:2])
                    row_kab = df_kabkota[df_kabkota["kode"] == kode_kab_common]
                    if not row_kab.empty:
                        result["kabkota"], result["kode_kabkota"], result["score_kabkota"] = row_kab.iloc[0]["nama"], kode_kab_common, DERIVED_SCORE
                        print(f"  -> kab/kota di-derive dari kodepos: {result['kabkota']}")
                if len(common) >= 1:
                    row_prov = df_prov[df_prov["kode"] == common[0]]
                    if not row_prov.empty:
                        result["provinsi"], result["kode_prov"], result["score_prov"] = row_prov.iloc[0]["nama"], common[0], DERIVED_SCORE
                        print(f"  -> provinsi di-derive dari kodepos: {result['provinsi']}")

                if not common:
                    print("  -> kandidat kodepos gak sharing prefix apapun, beneran kosongin")

            return result
        else:
            print(f"kodepos '{kodepos}' tidak ditemukan di tabel -> lanjut sebagai tanpa kodepos")

    # ============================================
    # PATH B: TANPA KODEPOS (atau kodepos gak valid)
    # ============================================
    result["match_path"] = "without_postcode"

    # LEVEL 1: PROVINSI (selalu global)
    nama_p, kode_p = exact_match_safe(tokens, df_prov, min_len=4)
    score_p = 100
    if not nama_p:
        nama_p, kode_p, score_p = fuzzy_match_one(leftover_text, df_prov, score_cutoff=STRICT_THRESHOLD)
    if nama_p:
        result["provinsi"], result["kode_prov"], result["score_prov"] = nama_p, kode_p, score_p
        print(f"provinsi ketemu: {nama_p} (skor={score_p})")
        leftover_text = remove_matched_text(leftover_text, nama_p)
        tokens = tokenize(expand_abbreviations(leftover_text))
    else:
        print("provinsi: tidak ketemu -> lanjut cari kab/kota secara GLOBAL")

    # LEVEL 2: KAB/KOTA -> persempit kalau provinsi ketemu, else global
    kandidat_kab = df_kabkota[df_kabkota["kode"].str.startswith(kode_p + ".")] if nama_p else df_kabkota
    nama_k, kode_k = exact_match_safe(tokens, kandidat_kab, min_len=4)
    score_k = 100
    if not nama_k:
        nama_k, kode_k, score_k = fuzzy_match_one(leftover_text, kandidat_kab, score_cutoff=STRICT_THRESHOLD)
    if nama_k:
        result["kabkota"], result["kode_kabkota"], result["score_kabkota"] = nama_k, kode_k, score_k
        print(f"kab/kota ketemu: {nama_k} (skor={score_k})")
        if not nama_p:
            kp_derived = kode_k.split(".")[0]
            row_prov = df_prov[df_prov["kode"] == kp_derived]
            if not row_prov.empty:
                result["provinsi"], result["kode_prov"], result["score_prov"] = row_prov.iloc[0]["nama"], kp_derived, score_k
                print(f"  -> provinsi di-derive dari kab/kota: {result['provinsi']}")
        leftover_text = remove_matched_text(leftover_text, nama_k)
        tokens = tokenize(expand_abbreviations(leftover_text))
    else:
        print("kab/kota: tidak ketemu -> lanjut cari kecamatan secara GLOBAL")

    # ---- CEK BIGRAM DESA DULU sebelum lock-in kecamatan ----
    # Ini nyegah token pendek kayak "PACE" ke-grab duluan jadi kecamatan,
    # padahal sebenernya bagian dari nama desa "PACEKULON" ("PACE KULON" di teks).
    desa_scope = df_desa[df_desa["kode"].str.startswith(kode_k + ".")] if nama_k else df_desa
    bigram_nama_d, bigram_kode_d = find_desa_bigram_match(tokens, desa_scope)

    if bigram_nama_d:
        print(f"desa ketemu duluan via bigram (sebelum cek kecamatan): {bigram_nama_d} (skor=100)")
        parts = bigram_kode_d.split(".")
        result["desa"], result["kode_desa"], result["score_desa"] = bigram_nama_d, bigram_kode_d, 100
        row_kec = df_kec[df_kec["kode"] == ".".join(parts[:3])]
        row_kab = df_kabkota[df_kabkota["kode"] == ".".join(parts[:2])]
        row_prov = df_prov[df_prov["kode"] == parts[0]]
        if not row_kec.empty:
            result["kecamatan"], result["kode_kec"], result["score_kec"] = row_kec.iloc[0]["nama"], ".".join(parts[:3]), 100
        if not result["kabkota"] and not row_kab.empty:
            result["kabkota"], result["kode_kabkota"], result["score_kabkota"] = row_kab.iloc[0]["nama"], ".".join(parts[:2]), 100
        if not result["provinsi"] and not row_prov.empty:
            result["provinsi"], result["kode_prov"], result["score_prov"] = row_prov.iloc[0]["nama"], parts[0], 100
        return result  # udah lengkap sampe desa, gak perlu proses level 3 & 4 lagi

    # LEVEL 3: KECAMATAN -> persempit kalau kab/kota ketemu, else global
    kandidat_kec = df_kec[df_kec["kode"].str.startswith(kode_k + ".")] if nama_k else df_kec
    nama_c, kode_c = exact_match_safe(tokens, kandidat_kec, min_len=4)
    score_c = 100
    if not nama_c:
        nama_c, kode_c, score_c = fuzzy_match_one(leftover_text, kandidat_kec, score_cutoff=STRICT_THRESHOLD)
    if nama_c:
        result["kecamatan"], result["kode_kec"], result["score_kec"] = nama_c, kode_c, score_c
        print(f"kecamatan ketemu: {nama_c} (skor={score_c})")
        if not nama_k:
            kk_derived = ".".join(kode_c.split(".")[:2])
            row_kab = df_kabkota[df_kabkota["kode"] == kk_derived]
            if not row_kab.empty:
                result["kabkota"], result["kode_kabkota"], result["score_kabkota"] = row_kab.iloc[0]["nama"], kk_derived, score_c
                print(f"  -> kab/kota di-derive dari kecamatan: {result['kabkota']}")
            if not result["provinsi"]:
                kp_derived = kode_c.split(".")[0]
                row_prov = df_prov[df_prov["kode"] == kp_derived]
                if not row_prov.empty:
                    result["provinsi"], result["kode_prov"], result["score_prov"] = row_prov.iloc[0]["nama"], kp_derived, score_c
                    print(f"  -> provinsi di-derive dari kecamatan: {result['provinsi']}")
        leftover_text = remove_matched_text(leftover_text, nama_c)
        tokens = tokenize(expand_abbreviations(leftover_text))
    else:
        print("kecamatan: tidak ketemu -> lanjut cari desa secara GLOBAL")

    # LEVEL 4: DESA -> persempit kalau kecamatan ketemu, else global
    kandidat_desa = df_desa[df_desa["kode"].str.startswith(kode_c + ".")] if nama_c else df_desa
    nama_d, kode_d = exact_match_safe(tokens, kandidat_desa, min_len=4)
    score_d = 100
    if not nama_d:
        nama_d, kode_d = find_desa_bigram_match(tokens, kandidat_desa)
        if nama_d:
            score_d = 100
    if not nama_d:
        nama_d, kode_d, score_d = fuzzy_match_one(leftover_text, kandidat_desa, score_cutoff=STRICT_THRESHOLD)
    if nama_d:
        result["desa"], result["kode_desa"], result["score_desa"] = nama_d, kode_d, score_d
        print(f"desa ketemu: {nama_d} (skor={score_d})")
        parts = kode_d.split(".")
        if not result["kecamatan"]:
            row = df_kec[df_kec["kode"] == ".".join(parts[:3])]
            if not row.empty:
                result["kecamatan"], result["kode_kec"], result["score_kec"] = row.iloc[0]["nama"], ".".join(parts[:3]), score_d
        if not result["kabkota"]:
            row = df_kabkota[df_kabkota["kode"] == ".".join(parts[:2])]
            if not row.empty:
                result["kabkota"], result["kode_kabkota"], result["score_kabkota"] = row.iloc[0]["nama"], ".".join(parts[:2]), score_d
        if not result["provinsi"]:
            row = df_prov[df_prov["kode"] == parts[0]]
            if not row.empty:
                result["provinsi"], result["kode_prov"], result["score_prov"] = row.iloc[0]["nama"], parts[0], score_d
    else:
        print("desa: tidak ketemu -> kosongin aja")

    return result


# test
print("=== case 1: tanpa kodepos, tanpa detail desa ===")
leftover_test1 = "64472 NGANJUK JAWA TIMUR"
print(match_wilayah_final(leftover_test1, kodepos=None))

print("\n=== case 2: dengan kodepos, tanpa detail desa (fallback ke common prefix) ===")
print(match_wilayah_final(leftover_test1, kodepos=64472))

print("\n=== case 3: tanpa kodepos, ADA nama desa 'pace kulon' (2 kata) ===")
leftover_test2 = "64472 NGANJUK JAWA TIMUR pace kulon jln. raya kediri nganjuk"
print(match_wilayah_final(leftover_test2, kodepos=None))

=== case 1: tanpa kodepos, tanpa detail desa ===
provinsi ketemu: JAWA TIMUR (skor=100)
kab/kota ketemu: KABUPATEN NGANJUK (skor=100)
kecamatan: tidak ketemu -> lanjut cari desa secara GLOBAL
desa: tidak ketemu -> kosongin aja
{'provinsi': 'JAWA TIMUR', 'kode_prov': '35', 'score_prov': 100, 'kabkota': 'KABUPATEN NGANJUK', 'kode_kabkota': '35.18', 'score_kabkota': 100, 'kecamatan': None, 'kode_kec': None, 'score_kec': 0, 'desa': None, 'kode_desa': None, 'score_desa': 0, 'match_path': 'without_postcode'}

=== case 2: dengan kodepos, tanpa detail desa (fallback ke common prefix) ===
kodepos '64472' -> 17 kandidat kode wilayah
desa gak ketemu -> coba derive kec/kab/prov dari kesamaan kandidat kodepos
  -> kecamatan di-derive dari kodepos: PACE
  -> kab/kota di-derive dari kodepos: KABUPATEN NGANJUK
  -> provinsi di-derive dari kodepos: JAWA TIMUR
{'provinsi': 'JAWA TIMUR', 'kode_prov': '35', 'score_prov': 90, 'kabkota': 'KABUPATEN NGANJUK', 'kode_kabkota': '35.18', 'score_kabkota': 90, 'ke

In [141]:
import re

STRICT_THRESHOLD = 98

# kata generik yang gak usah dipake buat nyari & buang token
GENERIC_PREFIXES = {"KABUPATEN", "KAB", "KOTA", "KECAMATAN", "KEC", "DESA", "KELURAHAN", "KEL"}


def remove_matched_text(text, matched_name):
    """
    Buang kata2 dari `matched_name` yang beneran muncul di `text`, per-kata (bukan frasa utuh).
    Fuzzy match sering matched candidate name-nya "KABUPATEN NGANJUK" padahal teks aslinya
    cuma ada kata "NGANJUK" doang -> kalau strip literal frasa utuh, gak ketemu & gak kehapus
    -> kata itu bisa "ke-reuse" di level berikutnya (double match).
    """
    if not matched_name:
        return text
    words = [w for w in re.split(r'\s+', matched_name.strip()) if w and w.upper() not in GENERIC_PREFIXES]
    cleaned = text
    for w in words:
        pattern = r'\b' + re.escape(w) + r'\b'
        cleaned = re.sub(pattern, ' ', cleaned, flags=re.IGNORECASE)
    return re.sub(r'\s+', ' ', cleaned).strip()


def common_kode_prefix(kode_list):
    """
    Cari level kode (prov/kab/kec/desa) yang SAMA di semua kandidat kode.
    len==1 -> cuma prov sama, len==2 -> prov+kab sama, len==3 -> +kec sama, dst.
    """
    splits = [k.split(".") for k in kode_list]
    max_level = min(len(s) for s in splits)
    common = []
    for i in range(max_level):
        vals = set(s[i] for s in splits)
        if len(vals) == 1:
            common.append(splits[0][i])
        else:
            break
    return common


def find_desa_bigram_match(tokens, desa_pool, min_len=6):
    """
    Cek apakah ada 2 token BERURUTAN di `tokens` yang kalau digabung tanpa spasi
    match EXACT ke salah satu nama desa di `desa_pool`.
    Nangkep kasus "PACE KULON" (2 kata di teks user) vs "PACEKULON" (1 kata di data),
    yang tanpa ini bakal ke-grab duluan sama level kecamatan (krn "PACE" match kecamatan).
    Return (nama_desa, kode_desa) atau (None, None).
    """
    if desa_pool.empty or len(tokens) < 2:
        return None, None
    names_map = {str(n).strip().upper(): (n, k) for n, k in zip(desa_pool["nama"], desa_pool["kode"])}
    for i in range(len(tokens) - 1):
        joined = (tokens[i] + tokens[i + 1]).upper()
        if len(joined) >= min_len and joined in names_map:
            return names_map[joined]
    return None, None


def match_wilayah_final(leftover_text, kodepos=None):
    # NORMALISASI: uppercase dari awal, krn tokenize() cuma nangkep [A-Z0-9]+.
    # Tanpa ini, teks lowercase/mixed-case bakal ke-skip total pas ditokenize.
    leftover_text = (leftover_text or "").strip().upper()

    tokens = tokenize(expand_abbreviations(leftover_text))
    result = {
        "provinsi": None, "kode_prov": None, "score_prov": 0,
        "kabkota": None, "kode_kabkota": None, "score_kabkota": 0,
        "kecamatan": None, "kode_kec": None, "score_kec": 0,
        "desa": None, "kode_desa": None, "score_desa": 0,
        "match_path": None,
    }

    # ============================================
    # PATH A: ADA KODEPOS
    # ============================================
    if kodepos:
        kandidat_kode = kodepos_df.loc[kodepos_df["kodepos"] == str(kodepos), "kode"].tolist()
        if kandidat_kode:
            print(f"kodepos '{kodepos}' -> {len(kandidat_kode)} kandidat kode wilayah")
            result["match_path"] = "with_postcode"

            pool_desa = df_desa[df_desa["kode"].isin(kandidat_kode)]
            nama_d, kode_d = exact_match_safe(tokens, pool_desa, min_len=4)
            score_d = 100
            if not nama_d:
                nama_d, kode_d = find_desa_bigram_match(tokens, pool_desa)
                if nama_d:
                    score_d = 100
                    print(f"desa ketemu via bigram: {nama_d} (skor={score_d})")
            if not nama_d:
                nama_d, kode_d, score_d = fuzzy_match_one(leftover_text, pool_desa, score_cutoff=STRICT_THRESHOLD)

            if nama_d:
                print(f"desa ketemu: {nama_d} (skor={score_d})")
                parts = kode_d.split(".")
                result["desa"], result["kode_desa"], result["score_desa"] = nama_d, kode_d, score_d
                row_kec = df_kec[df_kec["kode"] == ".".join(parts[:3])]
                row_kab = df_kabkota[df_kabkota["kode"] == ".".join(parts[:2])]
                row_prov = df_prov[df_prov["kode"] == parts[0]]
                if not row_kec.empty:
                    result["kecamatan"], result["kode_kec"], result["score_kec"] = row_kec.iloc[0]["nama"], ".".join(parts[:3]), score_d
                if not row_kab.empty:
                    result["kabkota"], result["kode_kabkota"], result["score_kabkota"] = row_kab.iloc[0]["nama"], ".".join(parts[:2]), score_d
                if not row_prov.empty:
                    result["provinsi"], result["kode_prov"], result["score_prov"] = row_prov.iloc[0]["nama"], parts[0], score_d
            else:
                # FALLBACK: desa gak ketemu, tapi kandidat_kode dari kodepos
                # kemungkinan besar semua satu kecamatan/kabkota/provinsi yang sama.
                print("desa gak ketemu -> coba derive kec/kab/prov dari kesamaan kandidat kodepos")
                common = common_kode_prefix(kandidat_kode)
                DERIVED_SCORE = 90  # penanda: derived dari postcode only, bukan exact/fuzzy match teks

                if len(common) >= 3:
                    kode_kec_common = ".".join(common[:3])
                    row_kec = df_kec[df_kec["kode"] == kode_kec_common]
                    if not row_kec.empty:
                        result["kecamatan"], result["kode_kec"], result["score_kec"] = row_kec.iloc[0]["nama"], kode_kec_common, DERIVED_SCORE
                        print(f"  -> kecamatan di-derive dari kodepos: {result['kecamatan']}")
                if len(common) >= 2:
                    kode_kab_common = ".".join(common[:2])
                    row_kab = df_kabkota[df_kabkota["kode"] == kode_kab_common]
                    if not row_kab.empty:
                        result["kabkota"], result["kode_kabkota"], result["score_kabkota"] = row_kab.iloc[0]["nama"], kode_kab_common, DERIVED_SCORE
                        print(f"  -> kab/kota di-derive dari kodepos: {result['kabkota']}")
                if len(common) >= 1:
                    row_prov = df_prov[df_prov["kode"] == common[0]]
                    if not row_prov.empty:
                        result["provinsi"], result["kode_prov"], result["score_prov"] = row_prov.iloc[0]["nama"], common[0], DERIVED_SCORE
                        print(f"  -> provinsi di-derive dari kodepos: {result['provinsi']}")

                if not common:
                    print("  -> kandidat kodepos gak sharing prefix apapun, beneran kosongin")

            return result
        else:
            print(f"kodepos '{kodepos}' tidak ditemukan di tabel -> lanjut sebagai tanpa kodepos")

    # ============================================
    # PATH B: TANPA KODEPOS (atau kodepos gak valid)
    # ============================================
    result["match_path"] = "without_postcode"

    # LEVEL 1: PROVINSI (selalu global)
    nama_p, kode_p = exact_match_safe(tokens, df_prov, min_len=4)
    score_p = 100
    if not nama_p:
        nama_p, kode_p, score_p = fuzzy_match_one(leftover_text, df_prov, score_cutoff=STRICT_THRESHOLD)
    if nama_p:
        result["provinsi"], result["kode_prov"], result["score_prov"] = nama_p, kode_p, score_p
        print(f"provinsi ketemu: {nama_p} (skor={score_p})")
        leftover_text = remove_matched_text(leftover_text, nama_p)
        tokens = tokenize(expand_abbreviations(leftover_text))
    else:
        print("provinsi: tidak ketemu -> lanjut cari kab/kota secara GLOBAL")

    # LEVEL 2: KAB/KOTA -> persempit kalau provinsi ketemu, else global
    kandidat_kab = df_kabkota[df_kabkota["kode"].str.startswith(kode_p + ".")] if nama_p else df_kabkota
    nama_k, kode_k = exact_match_safe(tokens, kandidat_kab, min_len=4)
    score_k = 100
    if not nama_k:
        nama_k, kode_k, score_k = fuzzy_match_one(leftover_text, kandidat_kab, score_cutoff=STRICT_THRESHOLD)
    if nama_k:
        result["kabkota"], result["kode_kabkota"], result["score_kabkota"] = nama_k, kode_k, score_k
        print(f"kab/kota ketemu: {nama_k} (skor={score_k})")
        if not nama_p:
            kp_derived = kode_k.split(".")[0]
            row_prov = df_prov[df_prov["kode"] == kp_derived]
            if not row_prov.empty:
                result["provinsi"], result["kode_prov"], result["score_prov"] = row_prov.iloc[0]["nama"], kp_derived, score_k
                print(f"  -> provinsi di-derive dari kab/kota: {result['provinsi']}")
        leftover_text = remove_matched_text(leftover_text, nama_k)
        tokens = tokenize(expand_abbreviations(leftover_text))
    else:
        print("kab/kota: tidak ketemu -> lanjut cari kecamatan secara GLOBAL")

    # ---- CEK BIGRAM DESA DULU sebelum lock-in kecamatan ----
    # Ini nyegah token pendek kayak "PACE" ke-grab duluan jadi kecamatan,
    # padahal sebenernya bagian dari nama desa "PACEKULON" ("PACE KULON" di teks).
    desa_scope = df_desa[df_desa["kode"].str.startswith(kode_k + ".")] if nama_k else df_desa
    bigram_nama_d, bigram_kode_d = find_desa_bigram_match(tokens, desa_scope)

    if bigram_nama_d:
        print(f"desa ketemu duluan via bigram (sebelum cek kecamatan): {bigram_nama_d} (skor=100)")
        parts = bigram_kode_d.split(".")
        result["desa"], result["kode_desa"], result["score_desa"] = bigram_nama_d, bigram_kode_d, 100
        row_kec = df_kec[df_kec["kode"] == ".".join(parts[:3])]
        row_kab = df_kabkota[df_kabkota["kode"] == ".".join(parts[:2])]
        row_prov = df_prov[df_prov["kode"] == parts[0]]
        if not row_kec.empty:
            result["kecamatan"], result["kode_kec"], result["score_kec"] = row_kec.iloc[0]["nama"], ".".join(parts[:3]), 100
        if not result["kabkota"] and not row_kab.empty:
            result["kabkota"], result["kode_kabkota"], result["score_kabkota"] = row_kab.iloc[0]["nama"], ".".join(parts[:2]), 100
        if not result["provinsi"] and not row_prov.empty:
            result["provinsi"], result["kode_prov"], result["score_prov"] = row_prov.iloc[0]["nama"], parts[0], 100
        return result  # udah lengkap sampe desa, gak perlu proses level 3 & 4 lagi

    # LEVEL 3: KECAMATAN -> persempit kalau kab/kota ketemu, else global
    kandidat_kec = df_kec[df_kec["kode"].str.startswith(kode_k + ".")] if nama_k else df_kec
    nama_c, kode_c = exact_match_safe(tokens, kandidat_kec, min_len=4)
    score_c = 100
    if not nama_c:
        nama_c, kode_c, score_c = fuzzy_match_one(leftover_text, kandidat_kec, score_cutoff=STRICT_THRESHOLD)
    if nama_c:
        result["kecamatan"], result["kode_kec"], result["score_kec"] = nama_c, kode_c, score_c
        print(f"kecamatan ketemu: {nama_c} (skor={score_c})")
        if not nama_k:
            kk_derived = ".".join(kode_c.split(".")[:2])
            row_kab = df_kabkota[df_kabkota["kode"] == kk_derived]
            if not row_kab.empty:
                result["kabkota"], result["kode_kabkota"], result["score_kabkota"] = row_kab.iloc[0]["nama"], kk_derived, score_c
                print(f"  -> kab/kota di-derive dari kecamatan: {result['kabkota']}")
            if not result["provinsi"]:
                kp_derived = kode_c.split(".")[0]
                row_prov = df_prov[df_prov["kode"] == kp_derived]
                if not row_prov.empty:
                    result["provinsi"], result["kode_prov"], result["score_prov"] = row_prov.iloc[0]["nama"], kp_derived, score_c
                    print(f"  -> provinsi di-derive dari kecamatan: {result['provinsi']}")
        leftover_text = remove_matched_text(leftover_text, nama_c)
        tokens = tokenize(expand_abbreviations(leftover_text))
    else:
        print("kecamatan: tidak ketemu -> lanjut cari desa secara GLOBAL")

    # LEVEL 4: DESA -> persempit kalau kecamatan ketemu, else global
    kandidat_desa = df_desa[df_desa["kode"].str.startswith(kode_c + ".")] if nama_c else df_desa
    nama_d, kode_d = exact_match_safe(tokens, kandidat_desa, min_len=4)
    score_d = 100
    if not nama_d:
        nama_d, kode_d = find_desa_bigram_match(tokens, kandidat_desa)
        if nama_d:
            score_d = 100
    if not nama_d:
        nama_d, kode_d, score_d = fuzzy_match_one(leftover_text, kandidat_desa, score_cutoff=STRICT_THRESHOLD)
    if nama_d:
        result["desa"], result["kode_desa"], result["score_desa"] = nama_d, kode_d, score_d
        print(f"desa ketemu: {nama_d} (skor={score_d})")
        parts = kode_d.split(".")
        if not result["kecamatan"]:
            row = df_kec[df_kec["kode"] == ".".join(parts[:3])]
            if not row.empty:
                result["kecamatan"], result["kode_kec"], result["score_kec"] = row.iloc[0]["nama"], ".".join(parts[:3]), score_d
        if not result["kabkota"]:
            row = df_kabkota[df_kabkota["kode"] == ".".join(parts[:2])]
            if not row.empty:
                result["kabkota"], result["kode_kabkota"], result["score_kabkota"] = row.iloc[0]["nama"], ".".join(parts[:2]), score_d
        if not result["provinsi"]:
            row = df_prov[df_prov["kode"] == parts[0]]
            if not row.empty:
                result["provinsi"], result["kode_prov"], result["score_prov"] = row.iloc[0]["nama"], parts[0], score_d
    else:
        print("desa: tidak ketemu -> kosongin aja")

    return result


# test
print("=== case 1: tanpa kodepos, tanpa detail desa ===")
print(match_wilayah_final("64472 NGANJUK JAWA TIMUR", kodepos=None))

print("\n=== case 2: dengan kodepos, tanpa detail desa (fallback ke common prefix) ===")
print(match_wilayah_final("64472 NGANJUK JAWA TIMUR", kodepos=64472))

print("\n=== case 3: tanpa kodepos, ADA nama desa 'pace kulon' (2 kata, lowercase) ===")
print(match_wilayah_final("64472 NGANJUK JAWA TIMUR pace kulon jln. raya kediri nganjuk", kodepos=None))

=== case 1: tanpa kodepos, tanpa detail desa ===
provinsi ketemu: JAWA TIMUR (skor=100)
kab/kota ketemu: KABUPATEN NGANJUK (skor=100)
kecamatan: tidak ketemu -> lanjut cari desa secara GLOBAL
desa: tidak ketemu -> kosongin aja
{'provinsi': 'JAWA TIMUR', 'kode_prov': '35', 'score_prov': 100, 'kabkota': 'KABUPATEN NGANJUK', 'kode_kabkota': '35.18', 'score_kabkota': 100, 'kecamatan': None, 'kode_kec': None, 'score_kec': 0, 'desa': None, 'kode_desa': None, 'score_desa': 0, 'match_path': 'without_postcode'}

=== case 2: dengan kodepos, tanpa detail desa (fallback ke common prefix) ===
kodepos '64472' -> 17 kandidat kode wilayah
desa gak ketemu -> coba derive kec/kab/prov dari kesamaan kandidat kodepos
  -> kecamatan di-derive dari kodepos: PACE
  -> kab/kota di-derive dari kodepos: KABUPATEN NGANJUK
  -> provinsi di-derive dari kodepos: JAWA TIMUR
{'provinsi': 'JAWA TIMUR', 'kode_prov': '35', 'score_prov': 90, 'kabkota': 'KABUPATEN NGANJUK', 'kode_kabkota': '35.18', 'score_kabkota': 90, 'ke

In [132]:
def show_postcode_detail(kodepos):
    kandidat_kode = kodepos_df.loc[kodepos_df["kodepos"] == str(kodepos), "kode"].tolist()
    print(f"kodepos '{kodepos}' -> {len(kandidat_kode)} kandidat kode")

    rows = []
    for kode in kandidat_kode:
        parts = kode.split(".")
        kode_kec = ".".join(parts[:3])
        kode_kab = ".".join(parts[:2])
        kode_prov = parts[0]

        nama_desa = df_desa.loc[df_desa["kode"] == kode, "nama"]
        nama_kec = df_kec.loc[df_kec["kode"] == kode_kec, "nama"]
        nama_kab = df_kabkota.loc[df_kabkota["kode"] == kode_kab, "nama"]
        nama_prov = df_prov.loc[df_prov["kode"] == kode_prov, "nama"]

        rows.append({
            "kode": kode,
            "desa": nama_desa.iloc[0] if not nama_desa.empty else None,
            "kecamatan": nama_kec.iloc[0] if not nama_kec.empty else None,
            "kabkota": nama_kab.iloc[0] if not nama_kab.empty else None,
            "provinsi": nama_prov.iloc[0] if not nama_prov.empty else None,
        })

    return pd.DataFrame(rows)

detail_df = show_postcode_detail("64472")
detail_df

kodepos '64472' -> 17 kandidat kode


,kode,desa,kecamatan,kabkota,provinsi
0,35.18.05.2001,JAMPES,PACE,KABUPATEN NGANJUK,JAWA TIMUR
1,35.18.05.2002,MLANDANGAN,PACE,KABUPATEN NGANJUK,JAWA TIMUR
2,35.18.05.2004,JATIGREGES,PACE,KABUPATEN NGANJUK,JAWA TIMUR
3,35.18.05.2005,JOHO,PACE,KABUPATEN NGANJUK,JAWA TIMUR
4,35.18.05.2006,SANAN,PACE,KABUPATEN NGANJUK,JAWA TIMUR
5,35.18.05.2007,PACEKULON,PACE,KABUPATEN NGANJUK,JAWA TIMUR
6,35.18.05.2008,CERME,PACE,KABUPATEN NGANJUK,JAWA TIMUR
7,35.18.05.2009,BABADAN,PACE,KABUPATEN NGANJUK,JAWA TIMUR
8,35.18.05.2010,BATEMBAT,PACE,KABUPATEN NGANJUK,JAWA TIMUR
9,35.18.05.2011,BANARAN,PACE,KABUPATEN NGANJUK,JAWA TIMUR


In [137]:
def show_postcode_detail(kodepos):
    kandidat_kode = kodepos_df.loc[kodepos_df["kodepos"] == str(kodepos), "kode"].tolist()
    print(f"kodepos '{kodepos}' -> {len(kandidat_kode)} kandidat kode")

    rows = []
    for kode in kandidat_kode:
        parts = kode.split(".")
        kode_kec = ".".join(parts[:3])
        kode_kab = ".".join(parts[:2])
        kode_prov = parts[0]

        nama_desa = df_desa.loc[df_desa["kode"] == kode, "nama"]
        nama_kec = df_kec.loc[df_kec["kode"] == kode_kec, "nama"]
        nama_kab = df_kabkota.loc[df_kabkota["kode"] == kode_kab, "nama"]
        nama_prov = df_prov.loc[df_prov["kode"] == kode_prov, "nama"]

        rows.append({
            "kode": kode,
            "desa": nama_desa.iloc[0] if not nama_desa.empty else None,
            "kecamatan": nama_kec.iloc[0] if not nama_kec.empty else None,
            "kabkota": nama_kab.iloc[0] if not nama_kab.empty else None,
            "provinsi": nama_prov.iloc[0] if not nama_prov.empty else None,
        })

    return pd.DataFrame(rows)

detail_df = show_postcode_detail("64472")
detail_df

kodepos '64472' -> 17 kandidat kode


,kode,desa,kecamatan,kabkota,provinsi
0,35.18.05.2001,JAMPES,PACE,KABUPATEN NGANJUK,JAWA TIMUR
1,35.18.05.2002,MLANDANGAN,PACE,KABUPATEN NGANJUK,JAWA TIMUR
2,35.18.05.2004,JATIGREGES,PACE,KABUPATEN NGANJUK,JAWA TIMUR
3,35.18.05.2005,JOHO,PACE,KABUPATEN NGANJUK,JAWA TIMUR
4,35.18.05.2006,SANAN,PACE,KABUPATEN NGANJUK,JAWA TIMUR
5,35.18.05.2007,PACEKULON,PACE,KABUPATEN NGANJUK,JAWA TIMUR
6,35.18.05.2008,CERME,PACE,KABUPATEN NGANJUK,JAWA TIMUR
7,35.18.05.2009,BABADAN,PACE,KABUPATEN NGANJUK,JAWA TIMUR
8,35.18.05.2010,BATEMBAT,PACE,KABUPATEN NGANJUK,JAWA TIMUR
9,35.18.05.2011,BANARAN,PACE,KABUPATEN NGANJUK,JAWA TIMUR


In [135]:
import os
import re

def match_address(text):
    if pd.isna(text):
        raw_text = ""
    else:
        raw_text = str(text)
    alamat_clean = re.sub(r"\s+", " ", raw_text.upper()).strip()

    remaining = alamat_clean
    kodepos, remaining = extract_and_remove_kodepos(remaining)
    rt, rw, remaining = extract_and_remove_rt_rw(remaining)
    no, remaining = extract_and_remove_no(remaining)
    lantai, remaining = extract_and_remove_lantai(remaining)
    gedung, remaining = extract_and_remove_gedung(remaining)
    jalan, remaining = extract_and_remove_jalan(remaining)
    blok, remaining = extract_and_remove_blok(remaining)
    kav, remaining = extract_and_remove_kav(remaining)

    leftover = clean_leftover(remaining)
    wilayah_result = match_wilayah_final(leftover, kodepos=kodepos)

    return {
        "alamat_raw": raw_text,
        "alamat_clean": alamat_clean,
        "kodepos": kodepos,
        "rt": rt,
        "rw": rw,
        "no": no,
        "lantai": lantai,
        "gedung": gedung,
        "jalan": jalan,
        "blok": blok,
        "kav": kav,
        "remaining": remaining,
        "leftover": leftover,
        **wilayah_result,
    }


master_csv_path = "/Users/tiarasabrina/Documents/PROJECT/AI/address-parsing/master-data/master-matched/Master_Matching_20260701_1059 (1)_matched.csv"
master_df = pd.read_csv(master_csv_path, usecols=["Alamat Lengkap"], nrows=50)
master_df["alamat_clean"] = master_df["Alamat Lengkap"].astype(str).str.upper().str.strip()

sample = master_df["alamat_clean"].dropna().head(50)
sample_results = sample.apply(match_address)
sample_df = pd.concat(
    [
        master_df.loc[sample.index, ["Alamat Lengkap", "alamat_clean"]],
        pd.DataFrame(sample_results.tolist(), index=sample.index),
    ],
    axis=1,
 )

sample_df

kodepos '12950' -> 1 kandidat kode wilayah
desa gak ketemu -> coba derive kec/kab/prov dari kesamaan kandidat kodepos
  -> kecamatan di-derive dari kodepos: SETIABUDI
  -> kab/kota di-derive dari kodepos: KOTA ADMINISTRASI JAKARTA SELATAN
  -> provinsi di-derive dari kodepos: DAERAH KHUSUS IBUKOTA JAKARTA
provinsi: tidak ketemu -> lanjut cari kab/kota secara GLOBAL
kab/kota: tidak ketemu -> lanjut cari kecamatan secara GLOBAL
kecamatan: tidak ketemu -> lanjut cari desa secara GLOBAL
desa: tidak ketemu -> kosongin aja
provinsi: tidak ketemu -> lanjut cari kab/kota secara GLOBAL
kab/kota: tidak ketemu -> lanjut cari kecamatan secara GLOBAL
kecamatan: tidak ketemu -> lanjut cari desa secara GLOBAL
desa: tidak ketemu -> kosongin aja
provinsi: tidak ketemu -> lanjut cari kab/kota secara GLOBAL
kab/kota: tidak ketemu -> lanjut cari kecamatan secara GLOBAL
kecamatan: tidak ketemu -> lanjut cari desa secara GLOBAL


KeyboardInterrupt: 